In [1]:
# ============================================================
# 1. LOAD DATASETS
# ============================================================

import pandas as pd
#iscx = pd.read_csv(r"/Users/sheeba/Downloads/consolidated_traffic_data.csv")
#darknet = pd.read_csv(r"/Users/sheeba/Desktop/ML PROJECT/Darknet.CSV")

iscx = pd.read_csv(r"C:\Users\LOQ\Projects\encrypted-traffic-classifier-xai\data\raw\consolidated_traffic_data.csv")

darknet = pd.read_csv(r"C:\Users\LOQ\Projects\encrypted-traffic-classifier-xai\data\raw\Darknet.CSV")

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print("\nISCX VPN-nonVPN:")
print("Shape:", iscx.shape)

print("\nCIC-Darknet2020:")
print("Shape:", darknet.shape)


DATASET OVERVIEW

ISCX VPN-nonVPN:
Shape: (59706, 24)

CIC-Darknet2020:
Shape: (158616, 85)


In [2]:
# ============================================================
# 2. DISPLAY COLUMN NAMES
# ============================================================

print("\n" + "=" * 70)
print("ISCX COLUMNS")
print("=" * 70)

for i, col in enumerate(iscx.columns, 1):
    print(f"{i}. {col}")

print("\n" + "=" * 70)
print("CIC-DARKNET2020 COLUMNS")
print("=" * 70)

for i, col in enumerate(darknet.columns, 1):
    print(f"{i}. {col}")



ISCX COLUMNS
1. duration
2. total_fiat
3. total_biat
4. min_fiat
5. min_biat
6. max_fiat
7. max_biat
8. mean_fiat
9. mean_biat
10. flowPktsPerSecond
11. flowBytesPerSecond
12. min_flowiat
13. max_flowiat
14. mean_flowiat
15. std_flowiat
16. min_active
17. mean_active
18. max_active
19. std_active
20. min_idle
21. mean_idle
22. max_idle
23. std_idle
24. traffic_type

CIC-DARKNET2020 COLUMNS
1. Flow ID
2. Src IP
3. Src Port
4. Dst IP
5. Dst Port
6. Protocol
7. Timestamp
8. Flow Duration
9. Total Fwd Packet
10. Total Bwd packets
11. Total Length of Fwd Packet
12. Total Length of Bwd Packet
13. Fwd Packet Length Max
14. Fwd Packet Length Min
15. Fwd Packet Length Mean
16. Fwd Packet Length Std
17. Bwd Packet Length Max
18. Bwd Packet Length Min
19. Bwd Packet Length Mean
20. Bwd Packet Length Std
21. Flow Bytes/s
22. Flow Packets/s
23. Flow IAT Mean
24. Flow IAT Std
25. Flow IAT Max
26. Flow IAT Min
27. Fwd IAT Total
28. Fwd IAT Mean
29. Fwd IAT Std
30. Fwd IAT Max
31. Fwd IAT Min
32. Bwd

In [3]:
# ============================================================
# 3. BASIC DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

print("\nISCX missing values:")
print(iscx.isnull().sum()[iscx.isnull().sum() > 0])

print("\nDarknet missing values:")
print(darknet.isnull().sum()[darknet.isnull().sum() > 0])




MISSING VALUES

ISCX missing values:
Series([], dtype: int64)

Darknet missing values:
Flow Bytes/s    48
dtype: int64


In [4]:
# ============================================================
# 4. CHECK FOR INFINITE VALUES
# ============================================================

print("\n" + "=" * 70)
print("INFINITE VALUES")
print("=" * 70)

iscx_numeric = iscx.select_dtypes(include="number")
darknet_numeric = darknet.select_dtypes(include="number")

print("\nISCX infinite values:")
print(iscx_numeric.isin([float("inf"), float("-inf")]).sum().sum())

print("\nDarknet infinite values:")
print(darknet_numeric.isin([float("inf"), float("-inf")]).sum().sum())



INFINITE VALUES

ISCX infinite values:
0

Darknet infinite values:
52


In [5]:
# ============================================================
# 1B. CHECK FOR EXTREME VALUES IN DARKNET
# ============================================================

darknet_numeric = darknet.select_dtypes(include="number")

print("=" * 70)
print("EXTREME VALUE CHECK")
print("=" * 70)

extreme_summary = pd.DataFrame({
    "min": darknet_numeric.min(),
    "max": darknet_numeric.max(),
    "median": darknet_numeric.median()
})

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 10)

print(extreme_summary.sort_values("max", ascending=False).head(25))

EXTREME VALUE CHECK
                                    min           max        median
Flow Bytes/s                   0.000000           inf  7.032077e+01
Flow Packets/s                 0.016669           inf  7.643558e+00
Idle Max                       0.000000  1.456417e+15  1.427909e+15
Idle Mean                      0.000000  1.456417e+15  7.281651e+14
Idle Min                       0.000000  1.456417e+15  5.664990e+06
Idle Std                       0.000000  1.029836e+15  0.000000e+00
Total Length of Fwd Packet     0.000000  7.693074e+08  4.400000e+01
Total Length of Bwd Packet     0.000000  6.704287e+08  0.000000e+00
Packet Length Variance         0.000000  1.591911e+08  8.533333e+01
Bwd Bulk Rate Avg              0.000000  1.515741e+08  0.000000e+00
Flow Duration                  0.000000  1.200000e+08  4.169220e+05
Fwd IAT Total                  0.000000  1.200000e+08  1.704945e+05
Bwd IAT Total                  0.000000  1.200000e+08  0.000000e+00
Fwd IAT Max                 

In [6]:
# ============================================================
# 1C. IDENTIFY TIMING-RELATED COLUMNS
# ============================================================

print("=" * 70)
print("ISCX TIMING-RELATED COLUMNS")
print("=" * 70)

iscx_timing_cols = [
    col for col in iscx.columns
    if any(word in col.lower() for word in
           ["duration", "iat", "active", "idle"])
]

print(iscx_timing_cols)


print("\n" + "=" * 70)
print("DARKNET TIMING-RELATED COLUMNS")
print("=" * 70)

darknet_timing_cols = [
    col for col in darknet.columns
    if any(word in col.lower() for word in
           ["duration", "iat", "active", "idle"])
]

print(darknet_timing_cols)

ISCX TIMING-RELATED COLUMNS
['duration', 'total_fiat', 'total_biat', 'min_fiat', 'min_biat', 'max_fiat', 'max_biat', 'mean_fiat', 'mean_biat', 'min_flowiat', 'max_flowiat', 'mean_flowiat', 'std_flowiat', 'min_active', 'mean_active', 'max_active', 'std_active', 'min_idle', 'mean_idle', 'max_idle', 'std_idle']

DARKNET TIMING-RELATED COLUMNS
['Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']


In [7]:
# ============================================================
# 1D. COMPARE CORRESPONDING TIMING FEATURES
# ============================================================
import numpy as np
timing_mapping = {
    "duration": "Flow Duration",
    "total_fiat": "Fwd IAT Total",
    "total_biat": "Bwd IAT Total",
    "min_fiat": "Fwd IAT Min",
    "min_biat": "Bwd IAT Min",
    "max_fiat": "Fwd IAT Max",
    "max_biat": "Bwd IAT Max",
    "mean_fiat": "Fwd IAT Mean",
    "mean_biat": "Bwd IAT Mean",
    "min_flowiat": "Flow IAT Min",
    "max_flowiat": "Flow IAT Max",
    "mean_flowiat": "Flow IAT Mean",
    "std_flowiat": "Flow IAT Std",
    "min_active": "Active Min",
    "mean_active": "Active Mean",
    "max_active": "Active Max",
    "std_active": "Active Std",
    "min_idle": "Idle Min",
    "mean_idle": "Idle Mean",
    "max_idle": "Idle Max",
    "std_idle": "Idle Std"
}

print("=" * 90)
print("TIMING FEATURE COMPARISON")
print("=" * 90)

for iscx_col, darknet_col in timing_mapping.items():

    iscx_values = pd.to_numeric(iscx[iscx_col], errors="coerce")
    darknet_values = pd.to_numeric(darknet[darknet_col], errors="coerce")

    iscx_values = iscx_values[np.isfinite(iscx_values)]
    darknet_values = darknet_values[np.isfinite(darknet_values)]

    print(f"\n{iscx_col:18s} <-> {darknet_col}")
    print(f"  ISCX    median: {iscx_values.median():.6g}")
    print(f"  Darknet median: {darknet_values.median():.6g}")
    print(f"  ISCX    99%ile: {iscx_values.quantile(0.99):.6g}")
    print(f"  Darknet 99%ile: {darknet_values.quantile(0.99):.6g}")

TIMING FEATURE COMPARISON

duration           <-> Flow Duration
  ISCX    median: 1.30665e+07
  Darknet median: 416922
  ISCX    99%ile: 1.19998e+08
  Darknet 99%ile: 1.1997e+08

total_fiat         <-> Fwd IAT Total
  ISCX    median: 1.16892e+07
  Darknet median: 170494
  ISCX    99%ile: 1.19983e+08
  Darknet 99%ile: 1.19963e+08

total_biat         <-> Bwd IAT Total
  ISCX    median: 1.02845e+07
  Darknet median: 0
  ISCX    99%ile: 1.19971e+08
  Darknet 99%ile: 1.199e+08

min_fiat           <-> Fwd IAT Min
  ISCX    median: 64
  Darknet median: 59
  ISCX    99%ile: 4.5204e+07
  Darknet 99%ile: 5.03416e+07

min_biat           <-> Bwd IAT Min
  ISCX    median: 20
  Darknet median: 0
  ISCX    99%ile: 4.5209e+07
  Darknet 99%ile: 4.50783e+07

max_fiat           <-> Fwd IAT Max
  ISCX    median: 537138
  Darknet median: 130298
  ISCX    99%ile: 9.88129e+07
  Darknet 99%ile: 8.29093e+07

max_biat           <-> Bwd IAT Max
  ISCX    median: 382847
  Darknet median: 0
  ISCX    99%ile: 6.030

In [8]:
# ============================================================
# 1E. INSPECT EXTREME IDLE-TIME ROWS
# ============================================================

idle_cols = [
    "Idle Min",
    "Idle Mean",
    "Idle Max",
    "Idle Std"
]

# Show rows with extremely large idle values
idle_extreme = darknet[
    (darknet["Idle Max"] > 1e12) |
    (darknet["Idle Mean"] > 1e12) |
    (darknet["Idle Std"] > 1e12)
]

print("=" * 70)
print("EXTREME IDLE-TIME ROWS")
print("=" * 70)

print("Number of affected rows:", len(idle_extreme))

print("\nIdle columns:")
print(idle_extreme[idle_cols].head(20))

print("\nLabels/categories of affected rows:")
print(idle_extreme.iloc[:, -5:].head(20))

EXTREME IDLE-TIME ROWS
Number of affected rows: 81330

Idle columns:
        Idle Min     Idle Mean      Idle Max      Idle Std
4   1.437765e+15  1.437765e+15  1.437765e+15  3.117718e+06
5   1.437765e+15  1.437765e+15  1.437765e+15  1.866111e+05
6   1.437765e+15  1.437765e+15  1.437765e+15  3.184630e+07
7   1.437765e+15  1.437765e+15  1.437765e+15  3.617028e+07
8   1.437765e+15  1.437765e+15  1.437765e+15  3.327790e+07
9   1.437765e+15  1.437765e+15  1.437765e+15  3.298254e+07
10  1.437765e+15  1.437765e+15  1.437765e+15  1.297022e+07
11  1.437765e+15  1.437765e+15  1.437765e+15  3.115338e+07
12  1.437765e+15  1.437765e+15  1.437765e+15  3.678758e+07
13  1.437765e+15  1.437765e+15  1.437765e+15  2.428393e+07
14  1.437765e+15  1.437765e+15  1.437765e+15  2.071678e+07
15  1.437765e+15  1.437765e+15  1.437765e+15  2.554571e+07
16  1.437765e+15  1.437765e+15  1.437765e+15  3.045667e+07
17  1.437765e+15  1.437765e+15  1.437765e+15  3.678432e+07
18  1.437765e+15  1.437765e+15  1.437765e+15  

In [9]:
# ============================================================
# 1G. CHECK WHETHER EXTREME IDLE VALUE IS A TIMESTAMP ARTIFACT
# ============================================================

extreme_idle_value = 1.437765e15

print("=" * 70)
print("EXTREME IDLE VALUE ANALYSIS")
print("=" * 70)

print("\nExact value frequency:")
print(
    darknet["Idle Max"]
    .value_counts()
    .head(10)
)

print("\nRows where Idle Min == Idle Mean == Idle Max:")
same_idle = darknet[
    (darknet["Idle Min"] == darknet["Idle Mean"]) &
    (darknet["Idle Mean"] == darknet["Idle Max"]) &
    (darknet["Idle Max"] > 1e12)
]

print("Count:", len(same_idle))

print("\nUnique extreme Idle Max values:")
print(
    darknet.loc[
        darknet["Idle Max"] > 1e12,
        "Idle Max"
    ].value_counts().head(20)
)

EXTREME IDLE VALUE ANALYSIS

Exact value frequency:
Idle Max
0.000000e+00    75218
1.456265e+15        4
1.456322e+15        4
1.432574e+15        3
1.432574e+15        3
1.427911e+15        3
1.456322e+15        2
1.456263e+15        2
1.456322e+15        2
1.456322e+15        2
Name: count, dtype: int64

Rows where Idle Min == Idle Mean == Idle Max:
Count: 9953

Unique extreme Idle Max values:
Idle Max
1.456322e+15    4
1.456265e+15    4
1.432574e+15    3
1.432574e+15    3
1.427911e+15    3
1.456324e+15    2
1.456324e+15    2
1.456324e+15    2
1.456324e+15    2
1.456324e+15    2
1.456324e+15    2
1.456323e+15    2
1.456322e+15    2
1.456324e+15    2
1.456324e+15    2
1.456324e+15    2
1.456324e+15    2
1.456322e+15    2
1.456324e+15    2
1.456324e+15    2
Name: count, dtype: int64


In [10]:
# ============================================================
# 1H. CHECK EXTREME IDLE VALUES AS POSSIBLE TIMESTAMPS
# ============================================================

extreme_idle = darknet[darknet["Idle Max"] > 1e12].copy()

print("=" * 70)
print("EXTREME IDLE VALUES — TIMESTAMP CHECK")
print("=" * 70)

for col in ["Idle Min", "Idle Mean", "Idle Max"]:
    print(f"\n{col}:")
    print(
        pd.to_datetime(
            extreme_idle[col].head(10),
            unit="us",
            errors="coerce"
        ).to_string()
    )

EXTREME IDLE VALUES — TIMESTAMP CHECK

Idle Min:
4    2015-07-24 19:09:45.151822
5    2015-07-24 19:10:00.509764
6    2015-07-24 19:09:45.216708
7    2015-07-24 19:09:46.283665
8    2015-07-24 19:09:48.997540
9    2015-07-24 19:10:00.508985
10   2015-07-24 19:09:49.677008
11   2015-07-24 19:10:06.055269
12   2015-07-24 19:09:55.070490
13   2015-07-24 19:10:01.598807

Idle Mean:
4    2015-07-24 19:09:50.528719
5    2015-07-24 19:10:00.593883
6    2015-07-24 19:10:17.950639
7    2015-07-24 19:10:35.467620
8    2015-07-24 19:10:21.340700
9    2015-07-24 19:11:02.825480
10   2015-07-24 19:09:52.474550
11   2015-07-24 19:10:32.510550
12   2015-07-24 19:10:40.125860
13   2015-07-24 19:11:16.673930

Idle Max:
4    2015-07-24 19:09:55.910109
5    2015-07-24 19:10:00.927702
6    2015-07-24 19:11:44.821766
7    2015-07-24 19:11:43.276183
8    2015-07-24 19:11:29.237655
9    2015-07-24 19:12:00.471714
10   2015-07-24 19:11:20.189255
11   2015-07-24 19:11:52.189400
12   2015-07-24 19:11:25.181282


In [11]:
# ============================================================
# 1Q. VERIFY THE EXACT 23-FEATURE ISCX → DARKNET MAPPING
# ============================================================

feature_mapping = {
    "duration": "Flow Duration",
    "total_fiat": "Fwd IAT Total",
    "total_biat": "Bwd IAT Total",
    "min_fiat": "Fwd IAT Min",
    "min_biat": "Bwd IAT Min",
    "max_fiat": "Fwd IAT Max",
    "max_biat": "Bwd IAT Max",
    "mean_fiat": "Fwd IAT Mean",
    "mean_biat": "Bwd IAT Mean",
    "flowPktsPerSecond": "Flow Packets/s",
    "flowBytesPerSecond": "Flow Bytes/s",
    "min_flowiat": "Flow IAT Min",
    "max_flowiat": "Flow IAT Max",
    "mean_flowiat": "Flow IAT Mean",
    "std_flowiat": "Flow IAT Std",
    "min_active": "Active Min",
    "mean_active": "Active Mean",
    "max_active": "Active Max",
    "std_active": "Active Std",
    "min_idle": "Idle Min",
    "mean_idle": "Idle Mean",
    "max_idle": "Idle Max",
    "std_idle": "Idle Std"
}

print("=" * 70)
print("ISCX → DARKNET: 23-FEATURE MAPPING")
print("=" * 70)

print(f"ISCX features:   {len(feature_mapping)}")
print(f"Darknet columns:  {len(set(feature_mapping.values()))}")

print("\nMapping:")
for iscx_col, darknet_col in feature_mapping.items():
    exists = darknet_col in darknet.columns
    print(f"{iscx_col:25s} -> {darknet_col:25s} {'✓' if exists else 'MISSING'}")

ISCX → DARKNET: 23-FEATURE MAPPING
ISCX features:   23
Darknet columns:  23

Mapping:
duration                  -> Flow Duration             ✓
total_fiat                -> Fwd IAT Total             ✓
total_biat                -> Bwd IAT Total             ✓
min_fiat                  -> Fwd IAT Min               ✓
min_biat                  -> Bwd IAT Min               ✓
max_fiat                  -> Fwd IAT Max               ✓
max_biat                  -> Bwd IAT Max               ✓
mean_fiat                 -> Fwd IAT Mean              ✓
mean_biat                 -> Bwd IAT Mean              ✓
flowPktsPerSecond         -> Flow Packets/s            ✓
flowBytesPerSecond        -> Flow Bytes/s              ✓
min_flowiat               -> Flow IAT Min              ✓
max_flowiat               -> Flow IAT Max              ✓
mean_flowiat              -> Flow IAT Mean             ✓
std_flowiat               -> Flow IAT Std              ✓
min_active                -> Active Min                ✓
me

In [12]:
# ============================================================
# 1V. TEST WHETHER CORRUPTED IDLE VALUES CAN BE RECONSTRUCTED
# ============================================================

check = darknet[darknet["Idle Max"] > 1e12].copy()

check["idle_span_us"] = (
    check["Idle Max"] - check["Idle Min"]
)

check["duration_diff_ratio"] = (
    abs(check["idle_span_us"] - check["Flow Duration"])
    / check["Flow Duration"].replace(0, np.nan)
)

print("=" * 80)
print("IDLE TIMESTAMP SPAN vs FLOW DURATION")
print("=" * 80)

valid = check["duration_diff_ratio"].notna()

print(f"Rows with timestamp-like Idle Max : {len(check)}")
print(f"Rows with valid Flow Duration     : {valid.sum()}")

for threshold in [0.01, 0.05, 0.10]:
    within = (
        check.loc[valid, "duration_diff_ratio"] <= threshold
    ).sum()

    percentage = within / valid.sum() * 100

    print(
        f"Within {threshold*100:.0f}% of Flow Duration : "
        f"{within} rows ({percentage:.2f}%)"
    )

IDLE TIMESTAMP SPAN vs FLOW DURATION
Rows with timestamp-like Idle Max : 81330
Rows with valid Flow Duration     : 81330
Within 1% of Flow Duration : 39355 rows (48.39%)
Within 5% of Flow Duration : 44576 rows (54.81%)
Within 10% of Flow Duration : 46124 rows (56.71%)


UNIT VERIFICATION

In [13]:
# UNIT VERIFICATION: Compare timing-feature scales between ISCX and Darknet

timing_features = {
    "duration": "Flow Duration",
    "total_fiat": "Fwd IAT Total",
    "total_biat": "Bwd IAT Total",
    "min_fiat": "Fwd IAT Min",
    "min_biat": "Bwd IAT Min",
    "max_fiat": "Fwd IAT Max",
    "max_biat": "Bwd IAT Max",
    "mean_fiat": "Fwd IAT Mean",
    "mean_biat": "Bwd IAT Mean",
    "min_flowiat": "Flow IAT Min",
    "max_flowiat": "Flow IAT Max",
    "mean_flowiat": "Flow IAT Mean",
    "std_flowiat": "Flow IAT Std",
    "min_active": "Active Min",
    "mean_active": "Active Mean",
    "max_active": "Active Max",
    "std_active": "Active Std",
    "min_idle": "Idle Min",
    "mean_idle": "Idle Mean",
    "max_idle": "Idle Max",
    "std_idle": "Idle Std"
}

print("Median and 99th-percentile comparison:\n")

for iscx_col, darknet_col in timing_features.items():
    iscx_median = iscx[iscx_col].median()
    darknet_median = darknet[darknet_col].median()

    iscx_99 = iscx[iscx_col].quantile(0.99)
    darknet_99 = darknet[darknet_col].quantile(0.99)

    print(
        f"{iscx_col:18} | "
        f"ISCX median: {iscx_median:.3g} | "
        f"Darknet median: {darknet_median:.3g} | "
        f"ISCX 99%: {iscx_99:.3g} | "
        f"Darknet 99%: {darknet_99:.3g}"
    )

Median and 99th-percentile comparison:

duration           | ISCX median: 1.31e+07 | Darknet median: 4.17e+05 | ISCX 99%: 1.2e+08 | Darknet 99%: 1.2e+08
total_fiat         | ISCX median: 1.17e+07 | Darknet median: 1.7e+05 | ISCX 99%: 1.2e+08 | Darknet 99%: 1.2e+08
total_biat         | ISCX median: 1.03e+07 | Darknet median: 0 | ISCX 99%: 1.2e+08 | Darknet 99%: 1.2e+08
min_fiat           | ISCX median: 64 | Darknet median: 59 | ISCX 99%: 4.52e+07 | Darknet 99%: 5.03e+07
min_biat           | ISCX median: 20 | Darknet median: 0 | ISCX 99%: 4.52e+07 | Darknet 99%: 4.51e+07
max_fiat           | ISCX median: 5.37e+05 | Darknet median: 1.3e+05 | ISCX 99%: 9.88e+07 | Darknet 99%: 8.29e+07
max_biat           | ISCX median: 3.83e+05 | Darknet median: 0 | ISCX 99%: 6.03e+07 | Darknet 99%: 7.63e+07
mean_fiat          | ISCX median: 1.02e+05 | Darknet median: 4.82e+04 | ISCX 99%: 4.52e+07 | Darknet 99%: 5.63e+07
mean_biat          | ISCX median: 5.39e+04 | Darknet median: 0 | ISCX 99%: 4.52e+07 | D

In [14]:
# FINAL DARKNET CLEANING

# Replace infinity with NaN
darknet_clean = darknet.replace([np.inf, -np.inf], np.nan).copy()

# Remove rows with missing values in the 23 mapped model features
darknet_clean = darknet_clean.dropna(subset=list(feature_mapping.values()))

print("Original Darknet rows :", len(darknet))
print("Cleaned Darknet rows  :", len(darknet_clean))
print("Rows removed          :", len(darknet) - len(darknet_clean))


Original Darknet rows : 158616
Cleaned Darknet rows  : 158566
Rows removed          : 50


In [15]:
# Remove corrupted Idle features from the cross-dataset feature set

idle_features = [
    "min_idle",
    "mean_idle",
    "max_idle",
    "std_idle"
]

cross_dataset_features = [
    f for f in feature_mapping.keys()
    if f not in idle_features
]

print("Features used for cross-dataset experiment:")
print(cross_dataset_features)
print("\nNumber of features:", len(cross_dataset_features))

Features used for cross-dataset experiment:
['duration', 'total_fiat', 'total_biat', 'min_fiat', 'min_biat', 'max_fiat', 'max_biat', 'mean_fiat', 'mean_biat', 'flowPktsPerSecond', 'flowBytesPerSecond', 'min_flowiat', 'max_flowiat', 'mean_flowiat', 'std_flowiat', 'min_active', 'mean_active', 'max_active', 'std_active']

Number of features: 19


In [16]:
# Create final Darknet feature matrix for the cross-dataset experiment

X_darknet = darknet_clean[
    [feature_mapping[f] for f in cross_dataset_features]
].copy()

print("X_darknet shape:", X_darknet.shape)
print("Missing values:", X_darknet.isna().sum().sum())
print("Infinite values:", np.isinf(X_darknet).sum().sum())

X_darknet shape: (158566, 19)
Missing values: 0
Infinite values: 0


In [17]:
# 7-class label mapping

print(darknet_clean["Label.1"].value_counts())

Label.1
P2P                48520
Browsing           46457
Audio-Streaming    19830
Chat               11623
File-Transfer      11089
Video-Streaming     9467
Email               6143
VOIP                3566
AUDIO-STREAMING     1506
Video-streaming      281
File-transfer         84
Name: count, dtype: int64


In [18]:
# Normalize Darknet labels into the 7-class taxonomy

label_mapping = {
    "AUDIO-STREAMING": "Audio-Streaming",
    "Audio-Streaming": "Audio-Streaming",

    "Video-streaming": "Video-Streaming",
    "Video-Streaming": "Video-Streaming",

    "File-transfer": "File-Transfer",
    "File-Transfer": "File-Transfer",

    "P2P": "P2P",
    "Browsing": "Browsing",
    "Chat": "Chat",
    "Email": "Email",
    "VOIP": "VOIP"
}

y_darknet = darknet_clean["Label.1"].map(label_mapping)

print(y_darknet.value_counts())
print("\nUnmapped labels:", y_darknet.isna().sum())

Label.1
P2P                48520
Browsing           46457
Audio-Streaming    21336
Chat               11623
File-Transfer      11173
Video-Streaming     9748
Email               6143
VOIP                3566
Name: count, dtype: int64

Unmapped labels: 0


In [19]:
# Encode the 8 traffic classes

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_darknet_encoded = label_encoder.fit_transform(y_darknet)

print("Classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

Classes: ['Audio-Streaming' 'Browsing' 'Chat' 'Email' 'File-Transfer' 'P2P' 'VOIP'
 'Video-Streaming']
Number of classes: 8


In [22]:
print(iscx.columns.tolist())

['duration', 'total_fiat', 'total_biat', 'min_fiat', 'min_biat', 'max_fiat', 'max_biat', 'mean_fiat', 'mean_biat', 'flowPktsPerSecond', 'flowBytesPerSecond', 'min_flowiat', 'max_flowiat', 'mean_flowiat', 'std_flowiat', 'min_active', 'mean_active', 'max_active', 'std_active', 'min_idle', 'mean_idle', 'max_idle', 'std_idle', 'traffic_type']


In [23]:
print("ISCX labels:")
print(iscx["traffic_type"].value_counts())

ISCX labels:
traffic_type
BROWSING         10000
VPN-BROWSING     10000
VOIP              6485
VPN-VOIP          5576
VPN-FT            4704
P2P               4000
FT                3975
VPN-P2P           3415
VPN-CHAT          2839
CHAT              2505
VPN-MAIL          2444
MAIL              1364
STREAMING         1284
VPN-STREAMING     1115
Name: count, dtype: int64


In [32]:
iscx_label_mapping = {
    "BROWSING": "Browsing",
    "VPN-BROWSING": "Browsing",
    "VOIP": "VOIP",
    "VPN-VOIP": "VOIP",
    "FT": "File-Transfer",
    "VPN-FT": "File-Transfer",
    "P2P": "P2P",
    "VPN-P2P": "P2P",
    "CHAT": "Chat",
    "VPN-CHAT": "Chat",
    "MAIL": "Email",
    "VPN-MAIL": "Email",
    "STREAMING": "Streaming",      # <-- fixed, was "Video-Streaming"
    "VPN-STREAMING": "Streaming"   # <-- fixed
}

iscx["mapped_label"] = iscx["traffic_type"].map(iscx_label_mapping)

print(iscx["mapped_label"].value_counts())
print("Unmapped:", iscx["mapped_label"].isna().sum())

y_iscx_common = iscx["mapped_label"].map(common_label_map)

mapped_label
Browsing         20000
VOIP             12061
File-Transfer     8679
P2P               7415
Chat              5344
Email             3808
Streaming         2399
Name: count, dtype: int64
Unmapped: 0


In [25]:
darknet_common_mapping = {
    "Audio-Streaming": "Streaming",
    "Video-Streaming": "Streaming",
    "Browsing": "Browsing",
    "Chat": "Chat",
    "Email": "Email",
    "File-Transfer": "File-Transfer",
    "P2P": "P2P",
    "VOIP": "VOIP"
}

darknet["common_label"] = darknet["Label"].map(darknet_common_mapping)

print("Darknet common labels:")
print(darknet["common_label"].value_counts())

Darknet common labels:
Series([], Name: count, dtype: int64)


In [26]:
print(darknet.columns.tolist())

['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',

In [27]:
print(darknet["Label"].value_counts())

Label
Non-Tor    110442
NonVPN      23863
VPN         22919
Tor          1392
Name: count, dtype: int64


In [28]:
print("Darknet traffic labels:")
print(darknet["Label.1"].value_counts())

Darknet traffic labels:
Label.1
P2P                48520
Browsing           46457
Audio-Streaming    19830
Chat               11629
File-Transfer      11098
Video-Streaming     9486
Email               6145
VOIP                3566
AUDIO-STREAMING     1520
Video-streaming      281
File-transfer         84
Name: count, dtype: int64


In [29]:
darknet_common_mapping = {
    "P2P": "P2P",
    "Browsing": "Browsing",
    "Audio-Streaming": "Streaming",
    "AUDIO-STREAMING": "Streaming",
    "Video-Streaming": "Streaming",
    "Video-streaming": "Streaming",
    "Chat": "Chat",
    "File-Transfer": "File-Transfer",
    "File-transfer": "File-Transfer",
    "Email": "Email",
    "VOIP": "VOIP"
}

darknet["common_label"] = darknet["Label.1"].map(darknet_common_mapping)

print("Darknet common labels:")
print(darknet["common_label"].value_counts())

print("\nUnmapped labels:")
print(darknet.loc[darknet["common_label"].isna(), "Label.1"].unique())

Darknet common labels:
common_label
P2P              48520
Browsing         46457
Streaming        31117
Chat             11629
File-Transfer    11182
Email             6145
VOIP              3566
Name: count, dtype: int64

Unmapped labels:
[]


In [30]:
# Define ONE fixed set of common classes, used identically for both datasets
common_classes = ["Browsing", "Chat", "Email", "File-Transfer", "P2P", "Streaming", "VOIP"]
common_label_map = {cls: i for i, cls in enumerate(common_classes)}

print("Common class → integer mapping:")
print(common_label_map)

Common class → integer mapping:
{'Browsing': 0, 'Chat': 1, 'Email': 2, 'File-Transfer': 3, 'P2P': 4, 'Streaming': 5, 'VOIP': 6}


In [31]:
darknet_common_mapping = {
    "P2P": "P2P",
    "Browsing": "Browsing",
    "Audio-Streaming": "Streaming",
    "AUDIO-STREAMING": "Streaming",
    "Video-Streaming": "Streaming",
    "Video-streaming": "Streaming",
    "Chat": "Chat",
    "File-Transfer": "File-Transfer",
    "File-transfer": "File-Transfer",
    "Email": "Email",
    "VOIP": "VOIP"
}

# IMPORTANT: apply this to darknet_clean (not the original darknet), to match X_darknet's rows
darknet_clean["common_label"] = darknet_clean["Label.1"].map(darknet_common_mapping)

print("Unmapped labels:", darknet_clean["common_label"].isna().sum())

y_darknet_common = darknet_clean["common_label"].map(common_label_map)

print(y_darknet_common.value_counts())

Unmapped labels: 0
common_label
4    48520
0    46457
5    31084
1    11623
3    11173
2     6143
6     3566
Name: count, dtype: int64


In [33]:
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Drop any rows where the label didn't map (shouldn't be any now, but just in case)
valid_idx = y_darknet_common.notna()
X_darknet_final = X_darknet[valid_idx]
y_darknet_final = y_darknet_common[valid_idx].astype(int)

X_darknet_train, X_darknet_test, y_darknet_train, y_darknet_test = train_test_split(
    X_darknet_final,
    y_darknet_final,
    test_size=0.20,
    stratify=y_darknet_final,
    random_state=42
)

rf_darknet = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_darknet.fit(X_darknet_train, y_darknet_train)

# Sanity check: evaluate on Darknet's own held-out test set first
y_pred_darknet = rf_darknet.predict(X_darknet_test)
print("RF trained on Darknet — Darknet test accuracy:", accuracy_score(y_darknet_test, y_pred_darknet))
print(classification_report(y_darknet_test, y_pred_darknet, target_names=common_classes))

RF trained on Darknet — Darknet test accuracy: 0.8436652582455698
               precision    recall  f1-score   support

     Browsing       0.84      0.94      0.88      9292
         Chat       0.71      0.67      0.69      2325
        Email       0.53      0.47      0.50      1228
File-Transfer       0.73      0.61      0.67      2235
          P2P       0.94      0.93      0.94      9704
    Streaming       0.88      0.83      0.86      6217
         VOIP       0.49      0.44      0.46       713

     accuracy                           0.84     31714
    macro avg       0.73      0.70      0.71     31714
 weighted avg       0.84      0.84      0.84     31714



In [36]:
# Select ISCX's 19 features (using ISCX column names)
X_iscx_cross = iscx[cross_dataset_features].copy()

# Rename columns to match the Darknet naming style the model was trained on
X_iscx_cross = X_iscx_cross.rename(columns=feature_mapping)

# Align labels, drop unmapped rows
valid_iscx_idx = y_iscx_common.notna()
X_iscx_cross = X_iscx_cross[valid_iscx_idx]
y_iscx_cross = y_iscx_common[valid_iscx_idx].astype(int)

print("X_iscx_cross shape:", X_iscx_cross.shape)
print("Columns:", X_iscx_cross.columns.tolist())
print("Missing values:", X_iscx_cross.isna().sum().sum())

X_iscx_cross shape: (59706, 19)
Columns: ['Flow Duration', 'Fwd IAT Total', 'Bwd IAT Total', 'Fwd IAT Min', 'Bwd IAT Min', 'Fwd IAT Max', 'Bwd IAT Max', 'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow Packets/s', 'Flow Bytes/s', 'Flow IAT Min', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Std', 'Active Min', 'Active Mean', 'Active Max', 'Active Std']
Missing values: 0


In [37]:
y_pred_iscx = rf_darknet.predict(X_iscx_cross)

print("RF trained on Darknet — ISCX test accuracy:", accuracy_score(y_iscx_cross, y_pred_iscx))
print(classification_report(y_iscx_cross, y_pred_iscx, target_names=common_classes))

RF trained on Darknet — ISCX test accuracy: 0.24027735905939102
               precision    recall  f1-score   support

     Browsing       0.34      0.15      0.21     20000
         Chat       0.28      0.23      0.25      5344
        Email       0.81      0.47      0.60      3808
File-Transfer       0.37      0.52      0.43      8679
          P2P       0.13      0.08      0.10      7415
    Streaming       0.06      0.59      0.10      2399
         VOIP       0.86      0.16      0.27     12061

     accuracy                           0.24     59706
    macro avg       0.41      0.31      0.28     59706
 weighted avg       0.44      0.24      0.26     59706



In [38]:
import os
import numpy as np

PROJECT_ROOT = os.path.abspath("..")
X_train_iscx_full = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "processed", "X_train.csv"))

# Reduce to the 19 shared features only
X_train_iscx_19 = X_train_iscx_full[cross_dataset_features].copy()

print("ISCX 19-feature training shape:", X_train_iscx_19.shape)

ISCX 19-feature training shape: (47764, 19)


In [40]:
import json

y_train_iscx_full = np.load(os.path.join(PROJECT_ROOT, "data", "processed", "y_train.npy"))

with open(os.path.join(PROJECT_ROOT, "data", "processed", "label_mapping.json")) as f:
    original_label_mapping = json.load(f)

int_to_original_label = {v: k for k, v in original_label_mapping.items()}
y_train_iscx_original_labels = [int_to_original_label[i] for i in y_train_iscx_full]

# Reuse the SAME iscx_label_mapping dict from Step 3 earlier (14-class -> 7-class names)
y_train_iscx_common_labels = [iscx_label_mapping[label] for label in y_train_iscx_original_labels]
y_train_iscx_19 = np.array([common_label_map[label] for label in y_train_iscx_common_labels])

print("Unique classes in training labels:", np.unique(y_train_iscx_19))

Unique classes in training labels: [0 1 2 3 4 5 6]


In [42]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf_iscx_cross = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_iscx_cross.fit(X_train_iscx_19, y_train_iscx_19)

xgb_iscx_cross = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=7,
    eval_metric="mlogloss", random_state=42
)
xgb_iscx_cross.fit(X_train_iscx_19, y_train_iscx_19)

print("RF and XGBoost (19-feature, cross-dataset) trained successfully!")

RF and XGBoost (19-feature, cross-dataset) trained successfully!


In [43]:
# Rename Darknet's 19 columns to ISCX-style names, since these models were trained on ISCX column names
X_darknet_for_forward = X_darknet_final.copy()
X_darknet_for_forward.columns = [
    k for k, v in feature_mapping.items() if v in X_darknet_final.columns
]

y_pred_darknet_rf = rf_iscx_cross.predict(X_darknet_for_forward)
y_pred_darknet_xgb = xgb_iscx_cross.predict(X_darknet_for_forward)

print("RF trained on ISCX (19-feat, cleaned) — Darknet test accuracy:",
      accuracy_score(y_darknet_final, y_pred_darknet_rf))
print(classification_report(y_darknet_final, y_pred_darknet_rf, target_names=common_classes))

print("\nXGBoost trained on ISCX (19-feat, cleaned) — Darknet test accuracy:",
      accuracy_score(y_darknet_final, y_pred_darknet_xgb))
print(classification_report(y_darknet_final, y_pred_darknet_xgb, target_names=common_classes))

RF trained on ISCX (19-feat, cleaned) — Darknet test accuracy: 0.213652359269957
               precision    recall  f1-score   support

     Browsing       0.47      0.42      0.44     46457
         Chat       0.10      0.04      0.06     11623
        Email       0.12      0.15      0.13      6143
File-Transfer       0.09      0.38      0.14     11173
          P2P       0.20      0.15      0.17     48520
    Streaming       0.19      0.02      0.04     31084
         VOIP       0.03      0.14      0.05      3566

     accuracy                           0.21    158566
    macro avg       0.17      0.19      0.15    158566
 weighted avg       0.25      0.21      0.21    158566


XGBoost trained on ISCX (19-feat, cleaned) — Darknet test accuracy: 0.17877098495263802
               precision    recall  f1-score   support

     Browsing       0.40      0.31      0.35     46457
         Chat       0.05      0.03      0.03     11623
        Email       0.19      0.09      0.12      6143
F

In [44]:
xgb_darknet = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=7,
    eval_metric="mlogloss", random_state=42
)
xgb_darknet.fit(X_darknet_train, y_darknet_train)

# Sanity check on Darknet's own test set
y_pred_darknet_self = xgb_darknet.predict(X_darknet_test)
print("XGBoost trained on Darknet — Darknet test accuracy:", accuracy_score(y_darknet_test, y_pred_darknet_self))

# The actual reverse cross-dataset test
y_pred_iscx_xgb = xgb_darknet.predict(X_iscx_cross)
print("\nXGBoost trained on Darknet — ISCX test accuracy:", accuracy_score(y_iscx_cross, y_pred_iscx_xgb))
print(classification_report(y_iscx_cross, y_pred_iscx_xgb, target_names=common_classes))

XGBoost trained on Darknet — Darknet test accuracy: 0.8197326102036955

XGBoost trained on Darknet — ISCX test accuracy: 0.19681439051351624
               precision    recall  f1-score   support

     Browsing       0.40      0.17      0.24     20000
         Chat       0.22      0.15      0.18      5344
        Email       0.78      0.37      0.50      3808
File-Transfer       0.31      0.36      0.33      8679
          P2P       0.13      0.11      0.12      7415
    Streaming       0.06      0.65      0.10      2399
         VOIP       0.85      0.05      0.09     12061

     accuracy                           0.20     59706
    macro avg       0.39      0.27      0.22     59706
 weighted avg       0.44      0.20      0.21     59706



In [45]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# Scale ISCX 19-feature training data
scaler_cross = StandardScaler()
X_train_iscx_19_scaled = scaler_cross.fit_transform(X_train_iscx_19.values)

# One-hot encode the 7-class labels
y_train_iscx_19_onehot = tf.keras.utils.to_categorical(y_train_iscx_19, num_classes=7)

# Reshape for CNN: (samples, 19, 1)
X_train_cnn_cross = X_train_iscx_19_scaled.reshape(X_train_iscx_19_scaled.shape[0], 19, 1)

cnn_cross = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(19, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(7, activation='softmax')
])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

cnn_cross.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn_cross.fit(
    X_train_cnn_cross, y_train_iscx_19_onehot,
    epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[early_stop]
)

print("CNN (19-feature, cross-dataset) trained successfully!")

Epoch 1/100


C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1195/1195 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.5144 - loss: 1.3504 - val_accuracy: 0.5874 - val_loss: 1.1343
Epoch 2/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5916 - loss: 1.1271 - val_accuracy: 0.6167 - val_loss: 1.0330
Epoch 3/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6195 - loss: 1.0488 - val_accuracy: 0.6413 - val_loss: 0.9806
Epoch 4/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6404 - loss: 1.0018 - val_accuracy: 0.6720 - val_loss: 0.9143
Epoch 5/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6595 - loss: 0.9588 - val_accuracy: 0.6861 - val_loss: 0.8915
Epoch 6/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6714 - loss: 0.9277 - val_accuracy: 0.6951 - val_loss: 0.8724
Epoch 7/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6843 - loss: 0.8957 - val_accuracy: 0.7062 - val_loss: 0.8511
Epoch 8/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6904 - loss: 0.8789 - val_

In [46]:

# Scale Darknet data using the SAME scaler fitted on ISCX training data
X_darknet_for_forward_scaled = scaler_cross.transform(X_darknet_for_forward.values)

# Reshape for CNN
X_darknet_cnn_forward = X_darknet_for_forward_scaled.reshape(X_darknet_for_forward_scaled.shape[0], 19, 1)

# Predict
cnn_pred_probs_forward = cnn_cross.predict(X_darknet_cnn_forward)
cnn_pred_forward = np.argmax(cnn_pred_probs_forward, axis=1)

print("CNN trained on ISCX (19-feat, cleaned) — Darknet test accuracy:",
      accuracy_score(y_darknet_final, cnn_pred_forward))
print(classification_report(y_darknet_final, cnn_pred_forward, target_names=common_classes))

4956/4956 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step
CNN trained on ISCX (19-feat, cleaned) — Darknet test accuracy: 0.24060643517525826
               precision    recall  f1-score   support

     Browsing       0.47      0.40      0.44     46457
         Chat       0.06      0.04      0.05     11623
        Email       0.05      0.11      0.07      6143
File-Transfer       0.06      0.19      0.09     11173
          P2P       0.30      0.32      0.31     48520
    Streaming       0.40      0.01      0.02     31084
         VOIP       0.04      0.11      0.06      3566

     accuracy                           0.24    158566
    macro avg       0.20      0.17      0.15    158566
 weighted avg       0.32      0.24      0.24    158566



In [47]:
from tensorflow.keras.layers import LSTM

# Reshape for LSTM: (samples, 1, 19) — single timestep, 19 features
X_train_lstm_cross = X_train_iscx_19_scaled.reshape(X_train_iscx_19_scaled.shape[0], 1, 19)

lstm_cross = Sequential([
    LSTM(64, input_shape=(1, 19), return_sequences=False),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(7, activation='softmax')
])

lstm_cross.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_cross.fit(
    X_train_lstm_cross, y_train_iscx_19_onehot,
    epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[early_stop]
)

print("LSTM (19-feature, cross-dataset) trained successfully!")

Epoch 1/100


C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1195/1195 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.5122 - loss: 1.3976 - val_accuracy: 0.5698 - val_loss: 1.2199
Epoch 2/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5628 - loss: 1.2328 - val_accuracy: 0.5794 - val_loss: 1.1539
Epoch 3/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5751 - loss: 1.1888 - val_accuracy: 0.6094 - val_loss: 1.1216
Epoch 4/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5876 - loss: 1.1537 - val_accuracy: 0.6160 - val_loss: 1.0913
Epoch 5/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6004 - loss: 1.1298 - val_accuracy: 0.6184 - val_loss: 1.0644
LSTM (19-feature, cross-dataset) trained successfully!


In [48]:
X_darknet_lstm_forward = X_darknet_for_forward_scaled.reshape(X_darknet_for_forward_scaled.shape[0], 1, 19)

lstm_pred_probs_forward = lstm_cross.predict(X_darknet_lstm_forward)
lstm_pred_forward = np.argmax(lstm_pred_probs_forward, axis=1)

print("LSTM trained on ISCX (19-feat, cleaned) — Darknet test accuracy:",
      accuracy_score(y_darknet_final, lstm_pred_forward))
print(classification_report(y_darknet_final, lstm_pred_forward, target_names=common_classes))

4956/4956 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step
LSTM trained on ISCX (19-feat, cleaned) — Darknet test accuracy: 0.32693641764312653
               precision    recall  f1-score   support

     Browsing       0.36      0.72      0.48     46457
         Chat       0.10      0.00      0.00     11623
        Email       0.03      0.05      0.04      6143
File-Transfer       0.10      0.04      0.05     11173
          P2P       0.38      0.35      0.37     48520
    Streaming       0.32      0.01      0.01     31084
         VOIP       0.08      0.12      0.10      3566

     accuracy                           0.33    158566
    macro avg       0.20      0.18      0.15    158566
 weighted avg       0.30      0.33      0.26    158566



In [49]:
# Scale Darknet training data with its own scaler
scaler_darknet_cross = StandardScaler()
X_darknet_train_scaled = scaler_darknet_cross.fit_transform(X_darknet_train.values)
X_darknet_test_scaled = scaler_darknet_cross.transform(X_darknet_test.values)

y_darknet_train_onehot = tf.keras.utils.to_categorical(y_darknet_train, num_classes=7)

X_darknet_train_cnn = X_darknet_train_scaled.reshape(X_darknet_train_scaled.shape[0], 19, 1)

cnn_darknet = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(19, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(7, activation='softmax')
])

cnn_darknet.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn_darknet.fit(
    X_darknet_train_cnn, y_darknet_train_onehot,
    epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[early_stop]
)

print("CNN trained on Darknet successfully!")

Epoch 1/100


C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


3172/3172 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.5114 - loss: 1.2924 - val_accuracy: 0.5825 - val_loss: 1.1527
Epoch 2/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.6015 - loss: 1.0862 - val_accuracy: 0.6464 - val_loss: 0.9906
Epoch 3/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.6552 - loss: 0.9757 - val_accuracy: 0.6589 - val_loss: 0.9441
Epoch 4/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.6749 - loss: 0.9236 - val_accuracy: 0.6977 - val_loss: 0.8663
Epoch 5/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.6878 - loss: 0.8980 - val_accuracy: 0.7017 - val_loss: 0.8470
CNN trained on Darknet successfully!


In [50]:
# Scale X_iscx_cross using the DARKNET-fitted scaler (since this model was trained on Darknet)
X_iscx_cross_scaled_for_darknet_model = scaler_darknet_cross.transform(X_iscx_cross.values)

# Reshape for CNN
X_iscx_cnn_reverse = X_iscx_cross_scaled_for_darknet_model.reshape(X_iscx_cross_scaled_for_darknet_model.shape[0], 19, 1)

cnn_pred_probs_reverse = cnn_darknet.predict(X_iscx_cnn_reverse)
cnn_pred_reverse = np.argmax(cnn_pred_probs_reverse, axis=1)

print("CNN trained on Darknet — ISCX test accuracy:", accuracy_score(y_iscx_cross, cnn_pred_reverse))
print(classification_report(y_iscx_cross, cnn_pred_reverse, target_names=common_classes))

1866/1866 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
CNN trained on Darknet — ISCX test accuracy: 0.05542156567179178
               precision    recall  f1-score   support

     Browsing       0.20      0.01      0.02     20000
         Chat       0.06      0.22      0.09      5344
        Email       0.00      0.00      0.00      3808
File-Transfer       0.00      0.00      0.00      8679
          P2P       0.00      0.00      0.00      7415
    Streaming       0.05      0.81      0.10      2399
         VOIP       0.00      0.00      0.00     12061

     accuracy                           0.06     59706
    macro avg       0.04      0.15      0.03     59706
 weighted avg       0.07      0.06      0.02     59706



C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [51]:
X_darknet_train_lstm = X_darknet_train_scaled.reshape(X_darknet_train_scaled.shape[0], 1, 19)

lstm_darknet = Sequential([
    LSTM(64, input_shape=(1, 19), return_sequences=False),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(7, activation='softmax')
])

lstm_darknet.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_darknet.fit(
    X_darknet_train_lstm, y_darknet_train_onehot,
    epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[early_stop]
)

print("LSTM trained on Darknet successfully!")

Epoch 1/100


C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3172/3172 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.4760 - loss: 1.3785 - val_accuracy: 0.5013 - val_loss: 1.2468
Epoch 2/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.5322 - loss: 1.2319 - val_accuracy: 0.5803 - val_loss: 1.1833
Epoch 3/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.5531 - loss: 1.1893 - val_accuracy: 0.5622 - val_loss: 1.1391
Epoch 4/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.5692 - loss: 1.1532 - val_accuracy: 0.5842 - val_loss: 1.1062
Epoch 5/100
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.5889 - loss: 1.1230 - val_accuracy: 0.6140 - val_loss: 1.0680
LSTM trained on Darknet successfully!


In [52]:
X_iscx_lstm_reverse = X_iscx_cross_scaled_for_darknet_model.reshape(X_iscx_cross_scaled_for_darknet_model.shape[0], 1, 19)

lstm_pred_probs_reverse = lstm_darknet.predict(X_iscx_lstm_reverse)
lstm_pred_reverse = np.argmax(lstm_pred_probs_reverse, axis=1)

print("LSTM trained on Darknet — ISCX test accuracy:", accuracy_score(y_iscx_cross, lstm_pred_reverse))
print(classification_report(y_iscx_cross, lstm_pred_reverse, target_names=common_classes))

1866/1866 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
LSTM trained on Darknet — ISCX test accuracy: 0.06153485411851405
               precision    recall  f1-score   support

     Browsing       0.27      0.02      0.03     20000
         Chat       0.01      0.00      0.00      5344
        Email       0.02      0.00      0.00      3808
File-Transfer       0.34      0.07      0.12      8679
          P2P       0.07      0.24      0.10      7415
    Streaming       0.03      0.39      0.06      2399
         VOIP       0.00      0.00      0.00     12061

     accuracy                           0.06     59706
    macro avg       0.11      0.10      0.05     59706
 weighted avg       0.15      0.06      0.04     59706



C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri